# Computing the probability distribution of ring parameters from observed stellar density
## Adapted to emcee MCMC from rejection sampling
### By Jorge I. Zuluaga, Jaime Alvarado-Montes, Sebastian Numpaque-Rodríguez and David Kipping

---

**Notebook original:** `Zuluaga_PhotoRing/mcra-planet-d.ipynb`  
**Notebook adaptado:** `analysis/emcee_analysis/mcra-planet-d_emcee.ipynb`  
**Planeta:** Kepler-51d  

---

### Exploraciones implementadas (idénticas al original):

1. **Free radius**: fe, Rplanet, ir, phir (`MR_simple_variance`)
2. **Fixed radius**: fe, ir, phir con Rplanet=Rp_min (`MR_simple_variance_NoRp`)
3. **Tau variation**: tau, fe, ir, phir con Rplanet=Rp_min (`MR_simple_variance_NoRp_tau`)

## Modules required

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('./dependencies')

from geotrans2 import *
from emcee_utils import (
    log_prior, log_likelihood, log_probability, log_probability_blobs,
    make_log_prob_func, initialize_walkers, samples_to_dataframe,
    compute_derived_quantities, convergence_summary
)

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import interp1d
import pickle
from scipy.stats import norm, multivariate_normal, gaussian_kde
from tqdm import tqdm
import warnings
import multiprocess as mp
from ipywidgets import interact, FloatSlider
import time
from copy import deepcopy
import os
import emcee

warnings.filterwarnings("ignore")

N_CPUS = mp.cpu_count()
print(f"CPUs disponibles: {N_CPUS}")

## Routines

In [ ]:
def plotSample(Xs, S, props, rho_range=(500,4000), chargs=dict(), csargs=dict(), prefix="noprefix"):
    """
    Función de visualización IDÉNTICA al notebook original.
    Usa PlotGrid de geotrans2.py para los corner plots.
    """
    # ################################################
    # TARGET DISTRIBUTIONS
    # ################################################
    fig1, axs = plt.subplots(1, 3, figsize=(12, 6))

    # First subplot
    axs[0].hist(Xs['rho_obs'], bins=30, density=True, alpha=0.5, label='MCMC Samples')
    x = np.linspace(500, 4000, 400)
    y = S.rho_obs_fun(x)
    axs[0].plot(x, y, 'r--', label='Interpolated Observed Density')
    y_true = S.rho_true_fun(x)
    axs[0].plot(x, y_true, 'b--', label='Interpolated True Density')
    axs[0].set_xlim(*rho_range)
    axs[0].set_xlabel(r'Observed Density [kg/m$^3$]')
    axs[0].set_ylabel('Frequency')
    axs[0].legend()

    # Second subplot
    axs[1].hist(Xs['delta'], bins=30, density=True, alpha=0.5, label='MCMC Samples')
    delta_values = np.linspace(Xs['delta'].min(), Xs['delta'].max(), 400)
    gaussian = S.delta_fun(delta_values)
    axs[1].plot(delta_values, gaussian, 'r--', label='Gaussian Distribution')
    axs[1].set_xlabel('Delta')
    axs[1].set_ylabel('Frequency')
    axs[1].legend()

    # Third subplot - bins=4 EXACTO como el original
    axs[2].hist(Xs['rho_true'], bins=4, density=True, alpha=0.5, label='MCMC Samples')
    x = np.linspace(500, 4000, 400)
    y = S.rho_true_fun(x)
    axs[2].plot(x, y, 'r--', label='Interpolated True Density')
    axs[2].set_xlim(*rho_range)
    axs[2].set_xlabel(r'True Density [kg/m$^3$]')
    axs[2].set_ylabel('Frequency')
    axs[2].legend()

    plt.tight_layout()
    plt.savefig(f"figures/target_distributions-{prefix}.png")
    plt.show()

    # ################################################
    # PARAMETER'S CONTOURS - Usando PlotGrid IDÉNTICO al original
    # ################################################
    columns = [prop for prop, vals in props.items()]
    data = np.array(Xs[columns])

    G = PlotGrid(props, figsize=3)

    hargs = dict(alpha=1, bins=10, density=False, colorbar=1, cmap="rainbow")
    hargs.update(chargs)
    G.plotHist(data, **hargs)

    sargs = dict(c='r', marker='.', s=2, edgecolors='none', alpha=0.3)
    sargs.update(csargs)
    G.scatterPlot(data, **sargs)
    G.fig.savefig(f"figures/corner_posteriors-{prefix}.png")

    # ################################################
    # PARAMETER'S POSTERIORS
    # ################################################
    fig2, axs = plt.subplots(1, len(props), figsize=(18, 6))

    for i, (prop, vals) in enumerate(props.items()):
        axs[i].hist(Xs[prop], bins=30, density=True, alpha=0.5, label=prop)
        axs[i].set_xlabel(prop)
        axs[i].set_ylabel('Frequency')
        axs[i].set_title(f'Histogram of {prop}')
        axs[i].legend()

    plt.tight_layout()
    plt.savefig(f"figures/parameters_posteriors-{prefix}.png")
    plt.show()

    return G, fig1, fig2


def get_maximum_kde(Xs, props):
    cols = props.keys()
    data_kde = Xs[cols].values.T
    kde = gaussian_kde(data_kde, bw_method=0.1)
    Xs['kde'] = kde(Xs[cols].T)
    max_kde_index = Xs['kde'].idxmax()
    max_kde_values = np.array(Xs.loc[max_kde_index, cols])
    peak_point = dict()
    for i, col in enumerate(cols):
        peak_point[col] = float(max_kde_values[i])
    return peak_point

## Extract system properties

### Load all densities

In [ ]:
with open('dependencies/rho_true_fun.pkl', 'rb') as f:
    rho_true_fun = pickle.load(f)

with open('dependencies/rho_obs_b_fun.pkl', 'rb') as f:
    rho_obs_b_fun = pickle.load(f)

with open('dependencies/rho_obs_c_fun.pkl', 'rb') as f:
    rho_obs_c_fun = pickle.load(f)

with open('dependencies/rho_obs_d_fun.pkl', 'rb') as f:
    rho_obs_d_fun = pickle.load(f)

print("Funciones de densidad cargadas correctamente.")

In [ ]:
x_range = np.linspace(700, 3000, 400)
plt.plot(x_range, rho_true_fun(x_range), label='True Stellar Density', color='black')
plt.plot(x_range, rho_obs_b_fun(x_range), label='Observed Density - Planet b', linestyle='--')
plt.plot(x_range, rho_obs_c_fun(x_range), label='Observed Density - Planet c', linestyle='-.')
plt.plot(x_range, rho_obs_d_fun(x_range), label='Observed Density - Planet d', linestyle=':')
plt.xlabel('Density')
plt.ylabel('Frequency')
plt.title('True and Observed Stellar Density Distributions')
plt.legend()
plt.show()

## Stellar grid

In [ ]:
df = pd.read_csv('dependencies/Kepler_51_Kipping/GKTHCatalog_Table4.csv')
k51 = df[df['id_starname'] == 'kic11773022']

rho_true_sun = MSUN/(4/3 * np.pi * RSUN**3)

M_mean = k51.iso_mass.values[0]
R_mean = k51.iso_rad.values[0]
M_std = (k51.iso_mass_err1.values[0] + abs(k51.iso_mass_err2.values[0]))/2
R_std = (k51.iso_rad_err1.values[0] + abs(k51.iso_rad_err2.values[0]))/2

Ms_mean = M_mean*MSUN
Ms_std = M_std*MSUN
Rs_mean = R_mean*RSUN
Rs_std = R_std*RSUN

print(f"Mass: {M_mean} +/- {M_std}")
print(f"Radius: {R_mean} +/- {R_std}")

Ng = 5
ts = 2
MS, RS = np.meshgrid(
    np.linspace(Ms_mean - ts*Ms_std, Ms_mean + ts*Ms_std, Ng)/MSUN,
    np.linspace(Rs_mean - ts*Rs_std, Rs_mean + ts*Rs_std, Ng)/RSUN
)
delta_Ms = MS[0,1] - MS[0,0]
delta_Rs = RS[1,0] - RS[0,0]

mean = [M_mean, R_mean]
rho_MR = -0.2
cov = [
    [M_std**2, rho_MR * M_std * R_std], 
    [rho_MR * M_std * R_std, R_std**2]
]
PS = np.zeros(MS.shape)
for i in range(MS.shape[0]):
    for j in range(RS.shape[1]):
        PS[i, j] = multivariate_normal.pdf([MS[i, j], RS[i, j]], mean=mean, cov=cov)
PS = PS*delta_Ms*delta_Rs

print(f"\nGrid estelar: {Ng}x{Ng} = {Ng**2} puntos")

## Sampling functions and parameters

In [ ]:
Nw = 8

def adjust_params(S, verbose=False):
    S.ap = ((GCONST * S.Mstar * S.Porb_mean**2) / (4 * pi**2))**(1/3)
    if verbose:
        print(f"Semimajor axis: {S.ap/AU:.4f} au = {S.ap/S.Rstar:.2f} Rs")
    S.iorb = np.arccos(S.borb_mean*S.Rstar/S.ap)
    if verbose:
        print(f"Orbital inclination: {S.iorb*RAD:.2f} degrees")
    S.updateSystem()

## Planet information

**Kepler-51d** - Datos de Masuda et al. 2024

In [ ]:
# https://exoplanetarchive.ipac.caltech.edu/overview/Kepler-51
# Planet d: Masuda et al. 2024

system_prefix = 'k51-planet_d'
rho_obs_fun = rho_obs_d_fun

# Planetary radius
Rp_mean = 0.831*RJUP  # Masuda et al. 2024
Rp_std = 0.016*RJUP  # Masuda et al. 2024

# Orbital period
Porb_mean = 130.1858*DAY  # Masuda et al. 2024
Porb_std = 0.0018*DAY  # Masuda et al. 2024

# Impact parameter
borb_mean = 0.0030  # Masuda et al. 2024
borb_std = 0.0950  # Masuda et al. 2024

# Planetary mass
Mp_mean = 0.021*MJUP
Mp_std = 0.003*MJUP

print(f"Planeta: Kepler-51d")
print(f"  Rp = {Rp_mean/RJUP:.4f} +/- {Rp_std/RJUP:.4f} Rjup")
print(f"  Porb = {Porb_mean/DAY:.5f} +/- {Porb_std/DAY:.5f} days")
print(f"  b = {borb_mean:.4f} +/- {borb_std:.4f}")
print(f"  Mp = {Mp_mean/MJUP:.4f} +/- {Mp_std/MJUP:.4f} Mjup")

### Build system

In [ ]:
delta_mean = (Rp_mean/Rs_mean)**2
delta_std = 2*delta_mean*(Rp_std/Rp_mean + Rs_std/Rs_mean)
print(f"Transit depth: {delta_mean*100:.2f} +/- {delta_std*100:.2f} %")

ap_mean = ((GCONST * Ms_mean * Porb_mean**2) / (4 * pi**2))**(1/3)
ap_std = 1/3*ap_mean*(Ms_std/Ms_mean + 2*Porb_std/Porb_mean)
print(f"Semimajor axis: {ap_mean/AU:.4f} +/- {ap_std/AU:.4f} au = {ap_mean/Rs_mean:.2f} Rs")

iorb_mean = np.arccos(borb_mean*Rs_mean/ap_mean)*RAD
iorb_std = np.sqrt((borb_std * Rs_mean / ap_mean)**2 + \
                   (borb_mean * Rs_std / ap_mean)**2 + \
                   (borb_mean * Rs_mean * ap_std / ap_mean**2)**2) * RAD
print(f"Orbital inclination: {iorb_mean:.2f} +/- {iorb_std:.2f} degrees")

Rp_min = REARTH*(Mp_mean/MEARTH)**(1/3)
fRp_min = Rp_min/Rp_mean
print(f"Minimum planet radius: {Rp_min/RJUP:.4f} RJup")
print(f"Minimum planet radius in terms of the ringless planet radius: {fRp_min:.4f} Rp")

System = RingedSystem(
    system = dict(
        Mstar=Ms_mean,
        Rstar=Rs_mean,
        Rplanet=Rp_mean,
        Mplanet=Mp_mean,
        ap=ap_mean,
        iorb=iorb_mean*DEG,
        fe=1,
        fi=1,
        ir=0.0*DEG,
        phir=0.0*DEG,
        tau=1.0,
    )
)

System.noauto = True

System.Porb_mean = Porb_mean
System.borb_mean = borb_mean
System.delta_mean = delta_mean
System.delta_std = delta_std
System.delta_fun = lambda x: norm.pdf(x, delta_mean, delta_std)
System.rho_obs_fun = rho_obs_fun
System.rho_true_fun = rho_true_fun

print()
print("After adjustment:")
adjust_params(System, True)

System.calculate_PR()
print(f"rho_obs: {System.rho_obs}, rho_true: {System.rho_true}, PR: {System.PR}")

---
# EXPLORATION 1: Parameters fe, Rp, ir, phir (Free radius)
---

In [ ]:
props = dict(
    fe = dict(
        label=r"$f_e$",
        range=[1.1, 6.0],
        scale=1,
    ),
    Rplanet = dict(
        label=r"$R_p$ [$R_{jup}$]",
        range=[fRp_min*Rp_mean/RJUP, Rp_mean/RJUP],
        scale=RJUP,
    ),
    ir = dict(
        label=r"$i_r$ [deg]",
        range=[0.0, 90.0],
        scale=DEG,
    ),
    phir = dict(
        label=r"$\phi_r$ [deg]",
        range=[0.0, 90.0],
        scale=DEG,
    ),
)

store_params = dict(
    rho_true = dict(prop='rho_true', scale=1),
    rho_obs = dict(prop='rho_obs', scale=1),
    PR = dict(prop='PR', scale=1),
    ieff = dict(prop='ieff', scale=RAD),
    teff = dict(prop='teff', scale=RAD),
    delta = dict(prop='Ar', scale=1/np.pi),
    Rstar = dict(prop='Rstar', scale=1/RSUN),
    Mstar = dict(prop='Mstar', scale=1/MSUN),
    ap = dict(prop='ap', scale=1/AU),
    ep = dict(prop='ep', scale=1),
    iorb = dict(prop='iorb', scale=RAD),
    Borb = dict(prop='Borb', scale=1),
    Porb = dict(prop='Porb', scale=1/DAY),
    tT = dict(prop='tT', scale=1),
    grazing = dict(prop='grazing', scale=1),
)

ndim = len(props)
print(f"Exploration 1: Free radius")
print(f"Dimensiones: {ndim}")
for name, vals in props.items():
    print(f"  {name}: [{vals['range'][0]:.4f}, {vals['range'][1]:.4f}]")

### emcee Configuration

In [ ]:
nwalkers = 32
nsteps_per_grid = 500
burn_in = 100
n_processes = min(8, N_CPUS)
Ns = int(1e4)

print(f"Configuración emcee:")
print(f"  nwalkers: {nwalkers}")
print(f"  nsteps_per_grid: {nsteps_per_grid}")
print(f"  burn_in: {burn_in}")
print(f"  n_processes: {n_processes}")
print(f"  Muestra objetivo: {Ns}")

### Sampling

In [ ]:
S = deepcopy(System)
S.tau = 1
S.fi = 1

sample_suffix = f"MR_simple_variance-N{Ns:.0e}"

columns = [prop for prop in props.keys()]
store_cols = list(store_params.keys())
all_samples = []

total_start_time = time.time()
grid_times = []

print(f"Iniciando muestreo MCMC con emcee...")
print(f"Grid estelar: {MS.shape[0]}x{MS.shape[1]} = {MS.shape[0]*MS.shape[1]} puntos")
print("="*60)

n = 0
for i in tqdm(range(MS.shape[0]), desc="Grid rows"):
    for j in range(RS.shape[1]):
        n += 1
        
        S.Mstar = MS[i, j] * MSUN
        S.Rstar = RS[i, j] * RSUN
        S.updateSystem()
        
        Np = int(PS[i, j] * Ns)
        if Np < 10:
            continue
        
        nsteps = max(50, int(Np / (nwalkers * (1 - burn_in/nsteps_per_grid))))
        
        grid_start = time.time()
        
        p0 = initialize_walkers(nwalkers, props, scatter=0.3, seed=n*42)
        log_prob_fn = make_log_prob_func(S, props, store_params, adjust_params)
        
        with mp.Pool(processes=n_processes) as pool:
            sampler = emcee.EnsembleSampler(
                nwalkers, ndim, log_prob_fn, pool=pool
            )
            sampler.run_mcmc(p0, nsteps, progress=False)
        
        burn = min(burn_in, nsteps // 2)
        flat_samples = sampler.get_chain(discard=burn, thin=1, flat=True)
        
        df_samples = pd.DataFrame(flat_samples, columns=columns)
        
        if len(df_samples) > Np:
            df_samples = df_samples.sample(n=Np, random_state=n)
        
        # CALCULAR CANTIDADES DERIVADAS INMEDIATAMENTE (con parámetros estelares correctos)
        for idx, row in df_samples.iterrows():
            theta = [row[col] for col in columns]
            _, spars = log_likelihood(theta, S, props, store_params, adjust_params)
            for col in store_cols:
                df_samples.loc[idx, col] = spars.get(col, np.nan)
        
        all_samples.append(df_samples)
        
        grid_end = time.time()
        grid_times.append(grid_end - grid_start)

Xs = pd.concat(all_samples, ignore_index=True)

total_end_time = time.time()
total_time = total_end_time - total_start_time

print("="*60)
print(f"Muestreo completado!")
print(f"  Muestras obtenidas: {len(Xs)}")
print(f"  Tiempo total: {total_time:.2f} s")

In [ ]:
output_path = f"tmp/ringed_sample-{system_prefix}-{sample_suffix}.csv"
Xs.to_csv(output_path, index=False)
print(f"\nResultados guardados en: {output_path}")

In [ ]:
Xs = pd.read_csv(f"tmp/ringed_sample-{system_prefix}-{sample_suffix}.csv")
plotSample(Xs, S, props, prefix=f"{system_prefix}-{sample_suffix}",
           csargs=dict(alpha=0), chargs=dict(bins=20))

### Summary statistics - Exploration 1

In [ ]:
from scipy.stats import gaussian_kde

# Variables de interés
key_columns = ['fe', 'Rplanet', 'ir', 'phir']
derived_columns = ['rho_obs', 'delta', 'rho_true', 'PR']

print("="*70)
print("EXPLORATION 1 STATISTICS (Free Radius)")
print("="*70)

for col in key_columns + derived_columns:
    if col in Xs.columns:
        data = Xs[col].dropna()
        percentiles = np.percentile(data, [16, 50, 84])
        print(f"{col}: {percentiles[1]:.4f} (+{percentiles[2]-percentiles[1]:.4f} / -{percentiles[1]-percentiles[0]:.4f})")
    else:
        print(f"{col}: NOT IN DATA")

### Maximum KDE point - Exploration 1

In [ ]:
from photoring import find_kde_maximum

best_params = find_kde_maximum(Xs, key_columns)
print("Maximum KDE point (Exploration 1 - Free Radius):")
for k, v in best_params.items():
    print(f"  {k}: {v:.4f}")

### Alternative plot (ieff, teff)

In [ ]:
altprops = dict(
    fe = props['fe'],
    Rplanet = props['Rplanet'],
    ieff = dict(
        label=r"$i_{\rm eff}$ [deg]",
        range=[0, 90],
    ),
    teff = dict(
        label=r"$\theta_{\rm eff}$ [deg]",
        range=[0, 90],
    )
)
plotSample(Xs, S, altprops, prefix=f"{system_prefix}-{sample_suffix}-veff", rho_range=(800, 3000),
           csargs=dict(alpha=0), chargs=dict(bins=20))

---
# EXPLORATION 2: Fixed radius (fe, ir, phir)
---

In [ ]:
props = dict(
    fe = dict(
        label=r"$f_e$",
        range=[1.1, 10.0],
        scale=1,
    ),
    ir = dict(
        active=1,
        label=r"$i_r$ [deg]",
        range=[0.0, 90.0],
        scale=DEG,
    ),
    phir = dict(
        active=1,
        label=r"$\phi_r$ [deg]",
        range=[0.0, 90.0],
        scale=DEG,
    ),
)

store_params = dict(
    rho_true = dict(prop='rho_true', scale=1),
    rho_obs = dict(prop='rho_obs', scale=1),
    PR = dict(prop='PR', scale=1),
    ieff = dict(prop='ieff', scale=RAD),
    teff = dict(prop='teff', scale=RAD),
    delta = dict(prop='Ar', scale=1/np.pi),
    Rplanet = dict(prop='Rplanet', scale=1/RJUP),
    Rstar = dict(prop='Rstar', scale=1/RSUN),
    Mstar = dict(prop='Mstar', scale=1/MSUN),
    ap = dict(prop='ap', scale=1/AU),
    ep = dict(prop='ep', scale=1),
    iorb = dict(prop='iorb', scale=RAD),
    Borb = dict(prop='Borb', scale=1),
    Porb = dict(prop='Porb', scale=1/DAY),
    tT = dict(prop='tT', scale=1),
    grazing = dict(prop='grazing', scale=1),
)

System.Rplanet = Rp_min

ndim = len(props)
print(f"Exploration 2: Fixed radius")
print(f"Rplanet fijo = {Rp_min/RJUP:.4f} Rjup")
print(f"Dimensiones: {ndim}")
for name, vals in props.items():
    print(f"  {name}: [{vals['range'][0]:.4f}, {vals['range'][1]:.4f}]")

### Sampling

In [ ]:
S = deepcopy(System)
S.tau = 1
S.fi = 1

sample_suffix = f"MR_simple_variance_NoRp-N{Ns:.0e}"

columns = [prop for prop in props.keys()]
store_cols = list(store_params.keys())
all_samples = []

total_start_time = time.time()
grid_times = []

print(f"Iniciando muestreo MCMC con emcee...")
print(f"Grid estelar: {MS.shape[0]}x{MS.shape[1]} = {MS.shape[0]*MS.shape[1]} puntos")
print("="*60)

n = 0
for i in tqdm(range(MS.shape[0]), desc="Grid rows"):
    for j in range(RS.shape[1]):
        n += 1
        
        S.Mstar = MS[i, j] * MSUN
        S.Rstar = RS[i, j] * RSUN
        S.updateSystem()
        
        Np = int(PS[i, j] * Ns)
        if Np < 10:
            continue
        
        nsteps = max(50, int(Np / (nwalkers * (1 - burn_in/nsteps_per_grid))))
        
        grid_start = time.time()
        
        p0 = initialize_walkers(nwalkers, props, scatter=0.3, seed=n*42)
        log_prob_fn = make_log_prob_func(S, props, store_params, adjust_params)
        
        with mp.Pool(processes=n_processes) as pool:
            sampler = emcee.EnsembleSampler(
                nwalkers, ndim, log_prob_fn, pool=pool
            )
            sampler.run_mcmc(p0, nsteps, progress=False)
        
        burn = min(burn_in, nsteps // 2)
        flat_samples = sampler.get_chain(discard=burn, thin=1, flat=True)
        
        df_samples = pd.DataFrame(flat_samples, columns=columns)
        
        if len(df_samples) > Np:
            df_samples = df_samples.sample(n=Np, random_state=n)
        
        # CALCULAR CANTIDADES DERIVADAS INMEDIATAMENTE (con parámetros estelares correctos)
        for idx, row in df_samples.iterrows():
            theta = [row[col] for col in columns]
            _, spars = log_likelihood(theta, S, props, store_params, adjust_params)
            for col in store_cols:
                df_samples.loc[idx, col] = spars.get(col, np.nan)
        
        all_samples.append(df_samples)
        
        grid_end = time.time()
        grid_times.append(grid_end - grid_start)

Xs = pd.concat(all_samples, ignore_index=True)

total_end_time = time.time()
total_time = total_end_time - total_start_time

print("="*60)
print(f"Muestreo completado!")
print(f"  Muestras obtenidas: {len(Xs)}")
print(f"  Tiempo total: {total_time:.2f} s")

In [ ]:
output_path = f"tmp/ringed_sample-{system_prefix}-{sample_suffix}.csv"
Xs.to_csv(output_path, index=False)
print(f"\nResultados guardados en: {output_path}")

In [ ]:
Xs = pd.read_csv(f"tmp/ringed_sample-{system_prefix}-{sample_suffix}.csv")
plotSample(Xs, S, props, prefix=f"{system_prefix}-{sample_suffix}",
           csargs=dict(alpha=0), chargs=dict(bins=20))

### Summary statistics - Exploration 2

In [ ]:
from scipy.stats import gaussian_kde

# Variables de interés
key_columns = ['fe', 'ir', 'phir']
derived_columns = ['rho_obs', 'delta', 'rho_true', 'PR']

print("="*70)
print("EXPLORATION 2 STATISTICS (Fixed Radius)")
print("="*70)

for col in key_columns + derived_columns:
    if col in Xs.columns:
        data = Xs[col].dropna()
        percentiles = np.percentile(data, [16, 50, 84])
        print(f"{col}: {percentiles[1]:.4f} (+{percentiles[2]-percentiles[1]:.4f} / -{percentiles[1]-percentiles[0]:.4f})")
    else:
        print(f"{col}: NOT IN DATA")

### Maximum KDE point - Exploration 2

In [ ]:
from photoring import find_kde_maximum

best_params = find_kde_maximum(Xs, key_columns)
print("Maximum KDE point (Exploration 2 - Fixed Radius):")
for k, v in best_params.items():
    print(f"  {k}: {v:.4f}")

---
# EXPLORATION 3: Tau variation (tau, fe, ir, phir)
---

In [ ]:
props = dict(
    tau = dict(
        label=r"$\tau$",
        range=[2.0, 10.0],
        scale=1,
    ),
    fe = dict(
        label=r"$f_e$",
        range=[1.1, 10.0],
        scale=1,
    ),
    ir = dict(
        active=1,
        label=r"$i_r$ [deg]",
        range=[0.0, 90.0],
        scale=DEG,
    ),
    phir = dict(
        label=r"$\phi_r$ [deg]",
        range=[0.0, 90.0],
        scale=DEG,
    ),
)

store_params = dict(
    rho_true = dict(prop='rho_true', scale=1),
    rho_obs = dict(prop='rho_obs', scale=1),
    PR = dict(prop='PR', scale=1),
    ieff = dict(prop='ieff', scale=RAD),
    teff = dict(prop='teff', scale=RAD),
    delta = dict(prop='Ar', scale=1/np.pi),
    Rplanet = dict(prop='Rplanet', scale=1/RJUP),
    Rstar = dict(prop='Rstar', scale=1/RSUN),
    Mstar = dict(prop='Mstar', scale=1/MSUN),
    ap = dict(prop='ap', scale=1/AU),
    ep = dict(prop='ep', scale=1),
    iorb = dict(prop='iorb', scale=RAD),
    Borb = dict(prop='Borb', scale=1),
    Porb = dict(prop='Porb', scale=1/DAY),
    tT = dict(prop='tT', scale=1),
    grazing = dict(prop='grazing', scale=1),
)

ndim = len(props)
print(f"Exploration 3: Tau variation")
print(f"Rplanet fijo = {Rp_min/RJUP:.4f} Rjup")
print(f"Dimensiones: {ndim}")
for name, vals in props.items():
    print(f"  {name}: [{vals['range'][0]:.4f}, {vals['range'][1]:.4f}]")

### Sampling

In [ ]:
S = deepcopy(System)
S.tau = 1
S.fi = 1

sample_suffix = f"MR_simple_variance_NoRp_tau-N{Ns:.0e}"

columns = [prop for prop in props.keys()]
store_cols = list(store_params.keys())
all_samples = []

total_start_time = time.time()
grid_times = []

print(f"Iniciando muestreo MCMC con emcee...")
print(f"Grid estelar: {MS.shape[0]}x{MS.shape[1]} = {MS.shape[0]*MS.shape[1]} puntos")
print("="*60)

n = 0
for i in tqdm(range(MS.shape[0]), desc="Grid rows"):
    for j in range(RS.shape[1]):
        n += 1
        
        S.Mstar = MS[i, j] * MSUN
        S.Rstar = RS[i, j] * RSUN
        S.updateSystem()
        
        Np = int(PS[i, j] * Ns)
        if Np < 10:
            continue
        
        nsteps = max(50, int(Np / (nwalkers * (1 - burn_in/nsteps_per_grid))))
        
        grid_start = time.time()
        
        p0 = initialize_walkers(nwalkers, props, scatter=0.3, seed=n*42)
        log_prob_fn = make_log_prob_func(S, props, store_params, adjust_params)
        
        with mp.Pool(processes=n_processes) as pool:
            sampler = emcee.EnsembleSampler(
                nwalkers, ndim, log_prob_fn, pool=pool
            )
            sampler.run_mcmc(p0, nsteps, progress=False)
        
        burn = min(burn_in, nsteps // 2)
        flat_samples = sampler.get_chain(discard=burn, thin=1, flat=True)
        
        df_samples = pd.DataFrame(flat_samples, columns=columns)
        
        if len(df_samples) > Np:
            df_samples = df_samples.sample(n=Np, random_state=n)
        
        # CALCULAR CANTIDADES DERIVADAS INMEDIATAMENTE (con parámetros estelares correctos)
        for idx, row in df_samples.iterrows():
            theta = [row[col] for col in columns]
            _, spars = log_likelihood(theta, S, props, store_params, adjust_params)
            for col in store_cols:
                df_samples.loc[idx, col] = spars.get(col, np.nan)
        
        all_samples.append(df_samples)
        
        grid_end = time.time()
        grid_times.append(grid_end - grid_start)

Xs = pd.concat(all_samples, ignore_index=True)

total_end_time = time.time()
total_time = total_end_time - total_start_time

print("="*60)
print(f"Muestreo completado!")
print(f"  Muestras obtenidas: {len(Xs)}")
print(f"  Tiempo total: {total_time:.2f} s")

In [ ]:
output_path = f"tmp/ringed_sample-{system_prefix}-{sample_suffix}.csv"
Xs.to_csv(output_path, index=False)
print(f"\nResultados guardados en: {output_path}")

In [ ]:
Xs = pd.read_csv(f"tmp/ringed_sample-{system_prefix}-{sample_suffix}.csv")
plotSample(Xs, S, props, prefix=f"{system_prefix}-{sample_suffix}",
           csargs=dict(alpha=0), chargs=dict(bins=20))

### Summary statistics - Exploration 3

In [ ]:
from scipy.stats import gaussian_kde

# Variables de interés
key_columns = ['fe', 'ir', 'phir', 'tau']
derived_columns = ['rho_obs', 'delta', 'rho_true', 'PR']

print("="*70)
print("EXPLORATION 3 STATISTICS (Tau variation, Fixed Radius)")
print("="*70)

for col in key_columns + derived_columns:
    if col in Xs.columns:
        data = Xs[col].dropna()
        percentiles = np.percentile(data, [16, 50, 84])
        print(f"{col}: {percentiles[1]:.4f} (+{percentiles[2]-percentiles[1]:.4f} / -{percentiles[1]-percentiles[0]:.4f})")
    else:
        print(f"{col}: NOT IN DATA")

### Maximum KDE point - Exploration 3

In [ ]:
from photoring import find_kde_maximum

best_params = find_kde_maximum(Xs, key_columns)
print("Maximum KDE point (Exploration 3 - Tau variation):")
for k, v in best_params.items():
    print(f"  {k}: {v:.4f}")

---

## Summary statistics

In [ ]:
print("Resumen estadístico de parámetros:")
print("="*60)

for col in props.keys():
    q16, q50, q84 = np.percentile(Xs[col], [16, 50, 84])
    err_low = q50 - q16
    err_high = q84 - q50
    print(f"{col:10s}: {q50:.4f} (+{err_high:.4f} / -{err_low:.4f})")

print("\nObservables derivados:")
print("="*60)
for col in ['rho_obs', 'rho_true', 'PR', 'delta', 'ieff', 'teff']:
    if col in Xs.columns:
        q16, q50, q84 = np.percentile(Xs[col].dropna(), [16, 50, 84])
        err_low = q50 - q16
        err_high = q84 - q50
        print(f"{col:10s}: {q50:.4f} (+{err_high:.4f} / -{err_low:.4f})")

## Maximum KDE point (best-fit)

In [ ]:
peak_point = get_maximum_kde(Xs, props)

print("Punto de máxima densidad KDE:")
for param, value in peak_point.items():
    print(f"  {param}: {value:.4f}")

---

## Fin del notebook

**Notebook original:** `Zuluaga_PhotoRing/mcra-planet-d.ipynb`  
**Notebook adaptado:** `analysis/emcee_analysis/mcra-planet-d_emcee.ipynb`